# Cortex Unified ML Gateway (Google Colab T4 GPU Edition)

This notebook runs all major ML workloads (LLM, Embeddings, Document Parsing) and exposes them through a **single static Ngrok domain**.
It is optimized for Google Colab's free T4 GPU tier.

> **⚠️ Before you run this:** the original version of this notebook had a live Ngrok auth token hardcoded in the config cell. That's been removed — the notebook now reads it from a Colab secret (or prompts you) instead. **If that token was ever shared, committed, or pasted anywhere, revoke it at https://dashboard.ngrok.com/authtokens and generate a new one** before using this notebook.

Other fixes in this version:
- vLLM startup is now polled via `/health` with a real timeout instead of a blind `time.sleep(15)`, and it fails fast with the log tail if the process crashes, instead of silently starting the gateway against a dead backend.
- Uploaded filenames in `/parse` are sanitized and temp files are cleaned up, instead of trusting the raw filename as a path.
- The `/v1/{path}` proxy strips hop-by-hop headers (`host`, `content-length`, etc.) before forwarding to vLLM, and returns a clean 503 instead of a stack trace if vLLM isn't reachable.
- The shared `httpx` client and the Ngrok tunnel are now closed on shutdown instead of leaking across re-runs.
- Added a `/health` endpoint on the gateway itself.

In [ ]:
!pip install -q pyngrok docling fastembed fastapi uvicorn python-multipart httpx vllm


In [ ]:
%%writefile cortex_ml_gateway.py
import uvicorn
import httpx
import os
import sys
import unicodedata
import re
import uuid
from pathlib import Path
from contextlib import asynccontextmanager
from fastapi import FastAPI, Request, UploadFile, File
from fastapi.responses import StreamingResponse, JSONResponse
from pyngrok import ngrok
from fastembed import TextEmbedding
from docling.document_converter import DocumentConverter
from docling.chunking import HierarchicalChunker

# Force unbuffered output for prints in this script
sys.stdout.reconfigure(line_buffering=True)

# Lazy-loaded models
embedding_model = None
doc_converter = None
chunker = None

# Shared HTTP client for proxying to the local vLLM server. Created/closed
# inside the lifespan (instead of at import time) so it's properly torn down.
vllm_client: httpx.AsyncClient | None = None

# Headers that must never be blindly forwarded between hops when proxying.
# Forwarding the original "host" header on to vLLM, or a stale
# "content-length"/"transfer-encoding", can confuse the backend or the
# streaming response.
HOP_BY_HOP_HEADERS = {
    "host", "content-length", "transfer-encoding", "connection",
    "keep-alive", "proxy-authenticate", "proxy-authorization",
    "te", "trailers", "upgrade",
}

@asynccontextmanager
async def lifespan(app: FastAPI):
    global embedding_model, doc_converter, chunker, vllm_client
    print("Loading FastEmbed...")
    embedding_model = TextEmbedding(model_name="BAAI/bge-base-en-v1.5", threads=4)
    print("Loading Docling...")
    doc_converter = DocumentConverter()
    print("Loading Hierarchical Chunker...")
    chunker = HierarchicalChunker()
    vllm_client = httpx.AsyncClient(base_url="http://localhost:8001", timeout=None)
    print("Unified Gateway Ready!")

    # Launch Ngrok Tunnel
    ngrok_token = os.getenv("NGROK_AUTH_TOKEN")
    ngrok_domain = os.getenv("NGROK_STATIC_DOMAIN")
    if ngrok_token and ngrok_domain:
        ngrok.set_auth_token(ngrok_token)
        # Clean the domain of https:// and trailing slashes to prevent PyngrokNgrokHTTPError 9038
        clean_domain = ngrok_domain.replace("https://", "").replace("http://", "").strip("/")
        public_url = ngrok.connect(8000, domain=clean_domain).public_url
        print("\n" + "="*60)
        print("\033[92mCORTEX ML GATEWAY IS LIVE!\033[0m")
        print("="*60)
        print(f"Add these variables to your Render .env file:")
        print(f"LLM_BASE_URL={public_url}/v1")
        print(f"FAST_MODEL_BASE_URL={public_url}/v1")
        print(f"EMBEDDING_MODEL_ENDPOINT={public_url}/v1")
        print(f"REMOTE_PARSER_URL={public_url}/parse")
        print("="*60 + "\n")
    else:
        print("WARNING: Ngrok Token or Domain not provided. Running locally only.")

    try:
        yield
    finally:
        # Clean shutdown: close the proxy client and tear down the tunnel
        # so re-running the notebook doesn't leak connections/tunnels.
        await vllm_client.aclose()
        ngrok.kill()

app = FastAPI(title="Cortex Unified ML Gateway", lifespan=lifespan)

@app.get("/health")
async def health():
    return {"status": "ok"}

# --- ROUTE 1: Docling Parsing & Chunking ---
@app.post("/parse")
async def parse_document(file: UploadFile = File(...)):
    print(f"[Docling] Parsing document: {file.filename}")
    file_bytes = await file.read()

    # Sanitize the filename: strip any directory components and prefix with
    # a random id, so a crafted filename like "../../etc/passwd" can't
    # escape /tmp, and two concurrent uploads with the same name can't
    # collide or overwrite each other.
    safe_name = Path(file.filename or "upload").name
    temp_path = f"/tmp/{uuid.uuid4().hex}_{safe_name}"
    with open(temp_path, "wb") as f:
        f.write(file_bytes)

    try:
        result = doc_converter.convert(temp_path)

        print(f"[Docling] Chunking document: {file.filename}")
        chunks_iter = chunker.chunk(result.document)

        artifact_chunks = []
        for index, chunk in enumerate(chunks_iter):
            chunk_text = chunk.text
            chunk_meta = chunk.meta.export_json_dict() if chunk.meta else {}

            headings = chunk_meta.get("headings", [])
            heading_path = "/".join(headings) if headings else "root"

            page_numbers = []
            for item in chunk_meta.get("doc_items", []):
                for prov in item.get("prov", []):
                    if "page_no" in prov:
                        page_numbers.append(str(prov["page_no"]))
            page_str = ",".join(sorted(set(page_numbers))) if page_numbers else "0"

            normalized_text = unicodedata.normalize("NFKC", chunk_text.strip())
            normalized_text = re.sub(r'\s+', ' ', normalized_text)

            artifact_chunks.append({
                "text": chunk_text,
                "headings": headings,
                "page_numbers": [int(p) for p in page_numbers] if page_numbers else [],
                "bbox": None,
                "chunk_index": index,
                "token_count": len(chunk_text.split()),
                "normalized_text": normalized_text,
                "heading_path": heading_path,
                "page_str": page_str
            })

        print(f"[Docling] Successfully processed {len(artifact_chunks)} chunks for {file.filename}")

        return {
            "markdown": result.document.export_to_markdown(),
            "metadata": {
                "origin": file.filename,
                "page_count": len(result.document.pages)
            },
            "page_count": len(result.document.pages),
            "chunks": artifact_chunks
        }
    finally:
        # Always clean up the temp file, even if parsing raised.
        try:
            os.remove(temp_path)
        except OSError:
            pass

# --- ROUTE 2: FastEmbed Embeddings (OpenAI Compatible) ---
@app.post("/v1/embeddings")
async def create_embeddings(request: Request):
    body = await request.json()
    texts = body.get("input", [])
    if isinstance(texts, str):
        texts = [texts]

    print(f"[FastEmbed] Start embedding {len(texts)} chunks")
    embeddings = list(embedding_model.embed(texts))
    print(f"[FastEmbed] Finished embedding {len(texts)} chunks")

    data = []
    for i, vec in enumerate(embeddings):
        data.append({
            "object": "embedding",
            "embedding": vec.tolist(),
            "index": i
        })

    return {
        "object": "list",
        "data": data,
        "model": "BAAI/bge-base-en-v1.5",
        "usage": {"prompt_tokens": 0, "total_tokens": 0}
    }

# --- ROUTE 3: Proxy LLM to Background vLLM ---
@app.api_route("/v1/{path:path}", methods=["GET", "POST", "PUT", "DELETE", "OPTIONS", "HEAD", "PATCH"])
async def proxy_vllm(path: str, request: Request):
    print(f"[vLLM Proxy] Routing /{path}")
    forward_headers = [
        (k, v) for k, v in request.headers.raw
        if k.decode("latin-1").lower() not in HOP_BY_HOP_HEADERS
    ]
    req = vllm_client.build_request(
        method=request.method,
        url=f"/v1/{path}",
        headers=forward_headers,
        content=await request.body()
    )
    try:
        response = await vllm_client.send(req, stream=True)
    except httpx.ConnectError:
        # vLLM isn't up yet (or crashed) — return a clear 503 instead of a
        # raw connection-refused stack trace bubbling up through FastAPI.
        return JSONResponse(
            status_code=503,
            content={"error": "vLLM backend is not reachable. It may still be loading the model, or it may have crashed — check vllm.log."}
        )
    return StreamingResponse(
        response.aiter_raw(),
        status_code=response.status_code,
        headers={k: v for k, v in response.headers.items() if k.lower() not in ('content-length', 'transfer-encoding')}
    )

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")


In [ ]:
import subprocess
import time
import sys
import os
import site
import glob
import httpx

# =========================================================================
# CONFIGURATION
# =========================================================================
# --- Ngrok credentials ---
# Do NOT hardcode your auth token in the notebook — anyone who sees this
# file (shared link, GitHub, copy/paste to an AI assistant, etc.) gets full
# access to your Ngrok account. Store it as a Colab secret instead:
# left sidebar -> key icon -> "Add new secret" -> name it NGROK_AUTH_TOKEN.
# NOTE: the token that was previously hardcoded here has been removed from
# this notebook. If it was ever shared or committed anywhere, revoke it at
# https://dashboard.ngrok.com/authtokens and generate a new one.
try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
except Exception:
    NGROK_AUTH_TOKEN = None

if not NGROK_AUTH_TOKEN:
    from getpass import getpass
    NGROK_AUTH_TOKEN = getpass("Enter your Ngrok auth token: ")

os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN
os.environ["NGROK_STATIC_DOMAIN"] = "https://entree-antiquely-rotting.ngrok-free.dev"  # your reserved static domain
os.environ["NGROK_DISABLE_CRL_CHECK"] = "true"

# =========================================================================
# LINK PIP-INSTALLED NVIDIA/CUDA RUNTIME LIBRARIES
# =========================================================================
# Colab's preinstalled system CUDA and the CUDA runtime that `pip install
# vllm` pulls in (as nvidia-*-cuXX wheel dependencies) don't always match.
# Those wheels ship their own .so files under site-packages/nvidia/*/lib,
# but the OS linker doesn't look there by default, which shows up as
# "libcudart.so.XX: cannot open shared object file" at import time. This
# just makes sure whatever CUDA runtime pip actually installed is
# discoverable — it's version-agnostic, so it's safe however pip resolves it.
site_packages = site.getsitepackages()[0]
nvidia_libs = glob.glob(f"{site_packages}/nvidia/*/lib")
current_ld = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_libs) + ":" + current_ld
print(f"Linked {len(nvidia_libs)} pip-installed NVIDIA library directories to the OS linker path.")

print("Starting vLLM Server in the background...")
llm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "--port", "8001",
    "--max-model-len", "16384",
    "--gpu-memory-utilization", "0.6",
    "--disable-custom-all-reduce",
    "--enforce-eager"
]
llm_process = subprocess.Popen(llm_cmd, stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)

# =========================================================================
# WAIT FOR vLLM TO ACTUALLY BE READY (instead of a fixed 15s sleep)
# =========================================================================
# A hardcoded `time.sleep(15)` isn't enough to download + load a 7B model on
# a cold Colab runtime, so the gateway would start proxying requests to a
# backend that isn't listening yet. It also hides real startup failures
# (OOM, a bad flag, an incompatible dependency) behind a silent wait. Poll
# the health endpoint instead, and fail fast with the log tail if the
# process dies.
print("Waiting for vLLM to become ready (can take several minutes on a cold start)...")
VLLM_READY_TIMEOUT = 600  # seconds
start = time.time()
ready = False
while time.time() - start < VLLM_READY_TIMEOUT:
    if llm_process.poll() is not None:
        print("\nERROR: vLLM process exited early. Last 40 lines of vllm.log:\n")
        with open("vllm.log") as f:
            print("".join(f.readlines()[-40:]))
        raise RuntimeError("vLLM failed to start — see log above.")
    try:
        r = httpx.get("http://localhost:8001/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except httpx.HTTPError:
        pass
    elapsed = int(time.time() - start)
    print(f"  ...still loading ({elapsed}s elapsed)", end="\r")
    time.sleep(5)

if not ready:
    print("\nERROR: vLLM did not become ready within the timeout. Last 40 lines of vllm.log:\n")
    with open("vllm.log") as f:
        print("".join(f.readlines()[-40:]))
    raise RuntimeError("vLLM readiness check timed out.")

print(f"\nvLLM is ready after {int(time.time() - start)}s.")
print("Starting Unified FastAPI Gateway (logs will stream below)...")
!python -u cortex_ml_gateway.py
